In [1]:
import pandas as pd
import numpy as np

from pathlib import Path, PurePath

from sklearn.cluster import KMeans
import sys

sys.path.append(str(Path.home()))
sys.path.append(r"C:\Users\bidali.KENYAGRANGE\OneDrive - ControlTech Limited\Documents\Projects\my_scripts")

import os
import seaborn as sns
import matplotlib.pyplot as plt

from mix_scripts import mix_api_utilities

In [2]:
#os.chdir(r'C:\Users\bidali.KENYAGRANGE\Documents\Projects\DA\Test')



period = 'January 2025'

In [3]:
#reports_df.groupby('report_name').groups

In [4]:
#for report_name, report_group in report_gbs:
#    print(report_name)

In [5]:
path = 'mix_report_template_za.xlsx'
reports_df = pd.read_excel(path, sheet_name='Reports')
reports_df = reports_df[reports_df.to_pull_umbrella > 0]
report_gbs = reports_df.groupby('report_name')
driver_reports = ['Total Kenya HV','Total Uganda']

Path('Results').mkdir(exist_ok=True)
for report_name, report_group in report_gbs:
    print(report_name)
    file_name = f'{report_name} Monthly Fleet Report {period}'
    report_result_file_name = Path(f'Results/{report_name}').joinpath(f'{file_name}.xlsx')
    if report_result_file_name.exists():
        continue
    organisations = report_group.report_dirs.unique()
    organisations = [Path(organisation) for organisation in organisations]
    trips_dirs = [organisation.joinpath('Trips') for organisation in organisations]
    events_dirs = [organisation.joinpath('Events') for organisation in organisations]
    trips_df_original = pd.concat([mix_api_utilities.read_mix_api_trips(trips_dir) for trips_dir in trips_dirs])
    events_df_original = pd.concat([mix_api_utilities.read_mix_api_events(events_dir) for events_dir in events_dirs])
    mix_api_utilities.write_bridge_events_df(events_df_original, report_name=report_name)
    events_df = mix_api_utilities.read_bridge_events_df(events_df_original, report_name=report_name)
    trips_df = trips_df_original.copy()
    FROM, TO = pd.to_datetime(pd.read_excel(organisations[0].joinpath('report_details.xlsx'), sheet_name='Time', index_col='Name'))
    
    #entity_column_name = ['ReportName', 'AssetDescription']
    #entity_column_name = ['AssetDescription', 'RegistrationNumber']
    #entity_column_name =  ['ReportName', 'DriverName']
    #entity_column_name = ['AssetDescription', 'DriverName']
    entity_column_name = ['DriverName']
    


    rag_score = mix_api_utilities.make_rag_score(trips_df, events_df, entity_column_name=entity_column_name)
    utilization = mix_api_utilities.make_daily_utilization(trips_df, FROM, TO)
    fuel_report = mix_api_utilities.make_fuel_report(trips_df)

    entity_column_name = 'DriverName'
    entity_column_name = 'AssetDescription'

    (top_n, top_n2) = mix_api_utilities.write_top_n(events_df, trips_df, entity_column_name = entity_column_name)
    
    path = Path(f'Results/{report_name}')
    path.mkdir(parents=True, exist_ok=True)
    writer = pd.ExcelWriter(path.joinpath(f'{file_name}.xlsx'))
    rag_score.to_excel(writer, sheet_name='Scoring')
    rag_score.to_excel(writer, sheet_name='Analysis')
    utilization.to_excel(writer, sheet_name='Utilization')
    fuel_report.to_excel(writer, sheet_name='Fuel')
    writer.close()

    writer2 = pd.ExcelWriter(path.joinpath('Top N.xlsx'))
    top_n.to_excel(writer2, sheet_name='Top1')
    top_n2.to_excel(writer2, sheet_name='Top2')
    writer2.close()
    ##
    distance_col_name = 'DistanceKilometers'
    entity_vehicle_trip = trips_df.pivot_table(values=distance_col_name,
                                 #index=[site_column_name, entity_column_name],
                                index=['ReportName', 'RegistrationNumber'],
                                aggfunc='sum')
    entity_vehicle_trip = entity_vehicle_trip.reset_index()
    entity_vehicle_trip.groupby('ReportName')[distance_col_name]\
            .describe().fillna(0).to_excel(f'Results/{report_name}/Trips_Summary.xlsx')
    ##
    count_col_name = 'TotalOccurances'
    #index=['SiteName', 'RegistrationNumber']
    index = ['SiteName', 'DriverName']
    entity_vehicle_event = events_df.pivot_table(values=count_col_name,
                                 index=index,
                                aggfunc='sum')

    entity_vehicle_event = entity_vehicle_event.reset_index()
    entity_vehicle_event.groupby('SiteName')[count_col_name]\
            .describe().fillna(0).to_excel(f'Results/{report_name}/Events_Summary2.xlsx')
    ##
    
    sites_pivot_events = events_df.pivot_table(values=count_col_name,
                                  columns='SiteName', index='EventName', aggfunc=sum)
    sites_pivot_events.fillna(0).T.to_excel(f'Results/{report_name}/Events vs Sites.xlsx')
    ##

Total Uganda
Ragos Total Ug\Trips
Bayusuf Total Uganda\Trips
Dikus Total Uganda\Trips
Roy T Total Uganda\Trips
Izwof Total Uganda\Trips
LOCAL - HARRY TRANSPORTERS\Trips
LOCAL - MUKASA COURIERS Total Uganda\Trips
MASS - SIBED Total Uganda\Trips
MASS - SUPER STAR FOWARDERS CO.LTD(SSF)\Trips
Skipping bridge write. File already exists at: Bridges\Total Uganda\bridge_violations.xlsx

